# Detecção de Objetos Cotidianos com YOLOv5
## Projeto 20 — Comparação entre resoluções de entrada (416×416 vs 640×640)

**Autor:** Mauro Gutemberg — IFPI Campus Corrente

**Dataset:** subconjunto de 10 classes cotidianas do COCO-2017
**Modelo:** YOLOv5s (transfer learning a partir dos pesos pré-treinados)
**Investigação experimental:** impacto da resolução de entrada sobre acurácia (mAP@0.5, mAP@0.5:0.95, Precisão, Recall, F1) e velocidade (FPS)

> ⚠️ **Execute em ambiente com GPU** (Google Colab: *Ambiente de execução → Alterar tipo → GPU T4*).
> Tempo estimado total: ~2–4 h dependendo do número de épocas e amostras.


## 1. Configuração do ambiente

Clonamos o repositório oficial do YOLOv5 e instalamos as dependências, além do **FiftyOne**, que usaremos para baixar apenas o subconjunto do COCO com as classes de interesse.

In [ ]:
import torch, os
print('PyTorch:', torch.__version__)
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Clonar YOLOv5 e instalar dependências
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -q -r requirements.txt
!pip install -q fiftyone

## 2. Download do subconjunto do COCO

### Justificativa da escolha das classes

O COCO completo possui 80 classes e mais de 118 mil imagens de treino, tornando o treinamento inviável no ambiente disponível. Conforme facultado pela especificação, selecionamos **10 classes de objetos genuinamente cotidianos**, seguindo três critérios:

1. **Aderência ao tema** — objetos presentes em cenas domésticas, urbanas e de escritório;
2. **Diversidade de escala** — objetos grandes (*person*, *car*), médios (*chair*, *dog*, *cat*, *bicycle*, *laptop*) e pequenos (*bottle*, *cup*, *cell phone*), permitindo analisar o efeito da resolução sobre objetos de diferentes tamanhos — o cerne da investigação experimental;
3. **Frequência no COCO** — classes bem representadas, garantindo exemplos suficientes mesmo com amostragem reduzida.

Amostramos **3000 imagens de treino** e **750 de validação** contendo essas classes.

In [ ]:
CLASSES = ['person', 'car', 'bicycle', 'dog', 'cat',
           'chair', 'bottle', 'cup', 'laptop', 'cell phone']

import fiftyone as fo
import fiftyone.zoo as foz

train_ds = foz.load_zoo_dataset(
    'coco-2017', split='train',
    label_types=['detections'], classes=CLASSES,
    max_samples=3000, shuffle=True, seed=42,
    dataset_name='coco10_train',
)
val_ds = foz.load_zoo_dataset(
    'coco-2017', split='validation',
    label_types=['detections'], classes=CLASSES,
    max_samples=750, shuffle=True, seed=42,
    dataset_name='coco10_val',
)
print(train_ds)
print(val_ds)

In [ ]:
import fiftyone.utils.random as four
from fiftyone import ViewField as F

EXPORT_DIR = '/content/coco10'

# Manter apenas as anotações das 10 classes de interesse
train_view = train_ds.filter_labels('ground_truth', F('label').is_in(CLASSES))
val_view   = val_ds.filter_labels('ground_truth', F('label').is_in(CLASSES))

for view, split in [(train_view, 'train'), (val_view, 'val')]:
    view.export(
        export_dir=EXPORT_DIR,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field='ground_truth',
        split=split,
        classes=CLASSES,
    )

print('Dataset exportado para', EXPORT_DIR)
!cat {EXPORT_DIR}/dataset.yaml

### Verificação do dataset

Contamos as instâncias por classe e visualizamos algumas imagens com suas *bounding boxes* (formato YOLO: `classe x_centro y_centro largura altura`, normalizados).

In [ ]:
import glob, collections, matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Contagem de instâncias por classe no treino
counts = collections.Counter()
for lbl in glob.glob(f'{EXPORT_DIR}/labels/train/*.txt'):
    with open(lbl) as f:
        for line in f:
            counts[CLASSES[int(line.split()[0])]] += 1

plt.figure(figsize=(10,4))
plt.bar(counts.keys(), counts.values(), color='seagreen')
plt.title('Instâncias por classe (treino)')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# Amostra visual com bounding boxes
imgs = sorted(glob.glob(f'{EXPORT_DIR}/images/train/*'))[:4]
fig, axes = plt.subplots(1, 4, figsize=(20,5))
for ax, img_path in zip(axes, imgs):
    im = Image.open(img_path); W, H = im.size
    ax.imshow(im); ax.axis('off')
    lbl_path = img_path.replace('/images/', '/labels/').rsplit('.',1)[0] + '.txt'
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            c, x, y, w, h = map(float, line.split())
            ax.add_patch(patches.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H,
                         fill=False, color='lime', lw=2))
            ax.text((x-w/2)*W, (y-h/2)*H-4, CLASSES[int(c)],
                    color='lime', fontsize=9, weight='bold')
plt.show()

## 3. Experimento 1 — Resolução 416 × 416

Treinamos o **YOLOv5s** com *transfer learning* (pesos `yolov5s.pt`), mantendo todos os hiperparâmetros padrão. **A única variável manipulada entre os experimentos é a resolução de entrada.**

In [ ]:
EPOCHS = 50
BATCH  = 16
DATA   = f'{EXPORT_DIR}/dataset.yaml'

!python train.py --img 416 --batch {BATCH} --epochs {EPOCHS} \
    --data {DATA} --weights yolov5s.pt \
    --name exp_416 --seed 42

## 4. Experimento 2 — Resolução 640 × 640

In [ ]:
!python train.py --img 640 --batch {BATCH} --epochs {EPOCHS} \
    --data {DATA} --weights yolov5s.pt \
    --name exp_640 --seed 42

## 5. Avaliação

Avaliamos os melhores pesos (`best.pt`) de cada experimento no conjunto de validação com o script oficial `val.py`, que aplica **non-maximum suppression** (IoU = 0.45) e reporta Precisão, Recall, mAP@0.5 e mAP@0.5:0.95, além dos tempos de pré-processamento, inferência e NMS — a partir dos quais calculamos o **FPS**.

In [ ]:
!python val.py --weights runs/train/exp_416/weights/best.pt \
    --data {DATA} --img 416 --iou-thres 0.45 \
    --name val_416 --verbose --save-txt

In [ ]:
!python val.py --weights runs/train/exp_640/weights/best.pt \
    --data {DATA} --img 640 --iou-thres 0.45 \
    --name val_640 --verbose --save-txt

### Extração automática das métricas

Executamos a validação novamente por API para capturar as métricas programaticamente e montar a tabela comparativa. O F1-score é calculado como média harmônica de Precisão e Recall, e o FPS a partir do tempo total por imagem (pré-processamento + inferência + NMS).

In [ ]:
import val as validate
from utils.general import check_dataset
import pandas as pd

def avaliar(pesos, imgsz, nome):
    resultados, maps, tempos = validate.run(
        data=DATA, weights=pesos, imgsz=imgsz,
        iou_thres=0.45, name=f'api_{nome}', verbose=True,
    )
    p, r, map50, map5095 = resultados[:4]
    f1 = 2 * p * r / (p + r + 1e-16)
    t_total_ms = sum(tempos)            # pré-proc + inferência + NMS (ms/imagem)
    fps = 1000.0 / t_total_ms
    return {
        'Experimento': nome,
        'Precisão': round(p, 4), 'Recall': round(r, 4),
        'F1-score': round(f1, 4),
        'mAP@0.5': round(map50, 4), 'mAP@0.5:0.95': round(map5095, 4),
        'Tempo/img (ms)': round(t_total_ms, 2), 'FPS': round(fps, 1),
    }, maps

res416, maps416 = avaliar('runs/train/exp_416/weights/best.pt', 416, '416x416')
res640, maps640 = avaliar('runs/train/exp_640/weights/best.pt', 640, '640x640')

df = pd.DataFrame([res416, res640]).set_index('Experimento')
df.to_csv('tabela_comparativa.csv')
print('\n===== TABELA COMPARATIVA =====')
df

In [ ]:
# mAP@0.5:0.95 por classe
df_classes = pd.DataFrame({
    'Classe': CLASSES,
    'mAP 416': [round(m, 4) for m in maps416],
    'mAP 640': [round(m, 4) for m in maps640],
})
df_classes['Δ (640 − 416)'] = (df_classes['mAP 640'] - df_classes['mAP 416']).round(4)
df_classes.to_csv('tabela_por_classe.csv', index=False)
df_classes

## 6. Interpretação dos resultados

Preencha esta seção após a execução, respondendo com base nas tabelas acima:

1. **Acurácia** — Qual resolução obteve maior mAP@0.5 e mAP@0.5:0.95? De quanto foi a diferença absoluta e relativa? A diferença foi maior no mAP@0.5:0.95 (localização mais precisa)?
2. **Objetos pequenos** — Na `tabela_por_classe.csv`, as classes pequenas (*bottle*, *cup*, *cell phone*) tiveram o maior ganho com 640×640? Isso confirma a hipótese de que resoluções maiores preservam detalhes espaciais críticos para objetos pequenos?
3. **Velocidade** — Qual foi a razão de FPS entre os experimentos? Ela se aproxima da razão teórica de custo computacional (640²/416² ≈ 2,37)?
4. **Conclusão prática** — Em que cenários você recomendaria cada configuração? (tempo real em hardware limitado vs análise offline com prioridade em acurácia)

**Resposta:** _(escreva aqui a sua análise)_

## 7. Exportação dos artefatos

Compactamos os resultados do experimento (métricas por época, pesos e tabelas) para anexar ao repositório.

In [ ]:
!zip -r resultados.zip \
    runs/train/exp_416/results.csv runs/train/exp_416/weights/best.pt \
    runs/train/exp_640/results.csv runs/train/exp_640/weights/best.pt \
    tabela_comparativa.csv tabela_por_classe.csv
print('Pronto! Baixe resultados.zip')